<h2>Midterm Python programming</h2>
This midterm of 4 topics: Python, pandas, SQL, and regex demonstrations be done in order to ensure that the set up code is run.  Taking the steps out of order within each unit may create unexpected results.  You may do the major units out of order.  Be sure to not be stuck on a problem and not get to later topics.  Manage your time. 
<p>
    <b>Be sure to run this first block with all the imports!</b>

In [93]:
#these usual imports should be sufficient
import psycopg2 as pg
import csv
import pandas as pd
from pandas import Series, DataFrame
import numpy as np  #just in case you refer to something useful from numpy.


In [94]:
# This is just to get another look at the data
mdem = pd.read_csv("/Users/Charley/Dropbox/Charley/Juniata/Data Science Masters/Data Acquisition & Visualization/Mplsdemo (1).csv")
mdem

,Unnamed: 0,neighborhood,population,white,black,foreignBorn,hhIncome,poverty,collegeGrad
0,1,Cedar Riverside,8247,0.353,0.464,0.408,18892,0.060,0.258
1,3,Phillips West,5184,0.199,0.538,0.318,18404,0.042,0.211
2,4,Downtown West,7141,0.561,0.211,0.203,67086,0.057,0.551
3,5,Downtown East,1674,0.543,0.221,0.221,70669,0.071,0.577
4,6,Shingle Creek,3249,0.407,0.259,0.140,59414,0.110,0.247
...,...,...,...,...,...,...,...,...,...
79,94,McKinley,3198,0.220,0.416,0.161,42556,0.101,0.122
80,95,Whittier,14604,0.549,0.187,0.215,35855,0.038,0.399
81,96,Lyndale,7441,0.515,0.226,0.285,38441,0.083,0.390
82,98,Columbia Park,1699,0.751,0.067,0.130,66545,0.058,0.418


<h3>PYTHON--CSV transformations</h3>
Follow the comments embedded below to make some simple transformations to the Minneapolis demographics data.
<ol>
    <li>Remove the first column of id numbers</li>
    <li>Make sure all column headers are lowercase</li>
    <li>Create a new column (npov) that estimates the number of people living in poverty (pop * poverty)</li>
   </ol>
   <p>Do not use Pandas here.  Just read a csv line, transform, write out the line.  Pandas is later.

In [95]:
#csv file function templates from your assignments


"""
You can edit this function to adjust anything about the header line
"""
def reviseHead(header, csvout):
    header.pop(0) #removal of idnumber (#1)
    
    #transform all the headers to lowercase. (#2) 
         #Either you hardcode specific ones or, better, you can write a general loop.
    #for h in....
    
    header[4]='foreignborn'
    header[5]='hhincome'
    header[7]='collegegrad'
    
    #add in the new column header (#3)
    header.append('npov')
    
    #write out the transformed header      
    csvout.writerow(header)

"""
You can edit this function to adjust the data line
"""
def reviseData(dataRec, csvout):
    dataRec.pop(0) #removal of idnumber (#1)
    
    
    #add a new element to calculate the number of people in poverty.  (#3)
    #round this number to the nearest integer.
    
    dataRec.append(round(float(dataRec[1])*float(dataRec[-2])))
    
    #write out the transformed data
    csvout.writerow(dataRec)

"""
This csvXform function is generic and should not need to be edited.
"""
def csvXform(inFile, outFile):
    inf = open(inFile,"r",encoding="utf-8")
    outf = open(outFile,"w", encoding="utf-8")
    csvinf = csv.reader(inf)
    csvout = csv.writer(outf)
    isheader = True
    for line in csvinf: #line is a list of values from a line of the file
        if isheader: #fix header info
            reviseHead(line,csvout)          
            isheader = False
        else:
            reviseData(line,csvout)
    inf.close()
    outf.close()
    

#invoke this code with the correct filenames containing the demographics data
csvXform('/Users/Charley/Dropbox/Charley/Juniata/Data Science Masters/Data Acquisition & Visualization/Mplsdemo (1).csv','/Users/Charley/Dropbox/Charley/Juniata/Data Science Masters/Data Acquisition & Visualization/Mplsdemo (1) Output.csv')


<h3>PANDAS tranformations</h3>
Follow the comments embedded below to make some simple transformations to the Minneapolis police stops data using Pandas.
<ol>
    <li>Input the data set into a dataframe</li>
    <li>Remove the (first) 4-5 digit "caseid" column</li>
    <li>Filter the rows for only the "MDC" entries in the MDC column</li>
    <li>Remove the MDC column</li>
    
   </ol>
    

In [96]:
mstop = pd.read_csv("/Users/Charley/Dropbox/Charley/Juniata/Data Science Masters/Data Acquisition & Visualization/MplsStops(1).csv")
mstop

,Unnamed: 0,idNum,date,problem,MDC,citationIssued,personSearch,vehicleSearch,preRace,race,gender,lat,long,policePrecinct,neighborhood
0,6823,17-000003,2017-01-01 00:00:42,suspicious,MDC,NaN,NO,NO,Unknown,Unknown,Unknown,44.966617,-93.246458,1,Cedar Riverside
1,6824,17-000007,2017-01-01 00:03:07,suspicious,MDC,NaN,NO,NO,Unknown,Unknown,Male,44.980450,-93.271340,1,Downtown West
2,6825,17-000073,2017-01-01 00:23:15,traffic,MDC,NaN,NO,NO,Unknown,White,Female,44.948350,-93.275380,5,Whittier
3,6826,17-000092,2017-01-01 00:33:48,suspicious,MDC,NaN,NO,NO,Unknown,East African,Male,44.948360,-93.281350,5,Whittier
4,6827,17-000098,2017-01-01 00:37:58,traffic,MDC,NaN,NO,NO,Unknown,White,Female,44.979078,-93.262076,1,Downtown West
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51915,60834,17-491442,2017-12-31 23:15:50,traffic,MDC,YES,NO,NO,Unknown,Black,Female,44.990547,-93.251285,2,Marcy Holmes
51916,60835,17-491445,2017-12-31 23:18:32,suspicious,MDC,NO,NO,NO,Unknown,Unknown,Unknown,44.959150,-93.277850,5,Whittier
51917,60836,17-491462,2017-12-31 23:31:57,traffic,MDC,NO,NO,NO,Unknown,Black,Male,44.997803,-93.252438,2,St. Anthony East
51918,60837,17-491480,2017-12-31 23:48:22,traffic,MDC,NO,YES,YES,Unknown,White,Male,44.989595,-93.252222,2,Marcy Holmes


In [97]:
df = pd.read_csv("/Users/Charley/Dropbox/Charley/Juniata/Data Science Masters/Data Acquisition & Visualization/MplsStops(1).csv", index_col=0) # (#1)
print(df.info()) #quick look at the number of values in the file

#remove first column (#2)
del df['idNum']

#filter for the MDC entries (#3)
mdcdf = df[df['MDC']=='MDC']

#remove the MDC column (#4)
del mdcdf['MDC']

#write the dataframe out to a csv file
mdcdf.to_csv("/Users/Charley/Dropbox/Charley/Juniata/Data Science Masters/Data Acquisition & Visualization/MplsStopsOutput.csv", header=True)


<class 'pandas.core.frame.DataFrame'>
Index: 51920 entries, 6823 to 60838
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   idNum           51920 non-null  object 
 1   date            51920 non-null  object 
 2   problem         51920 non-null  object 
 3   MDC             51920 non-null  object 
 4   citationIssued  19110 non-null  object 
 5   personSearch    43699 non-null  object 
 6   vehicleSearch   43699 non-null  object 
 7   preRace         43699 non-null  object 
 8   race            43699 non-null  object 
 9   gender          43638 non-null  object 
 10  lat             51920 non-null  float64
 11  long            51920 non-null  float64
 12  policePrecinct  51920 non-null  int64  
 13  neighborhood    51920 non-null  object 
dtypes: float64(2), int64(1), object(11)
memory usage: 5.9+ MB
None


Some simple pandas exploration operations

<h3>SQL queries</h3>
<p>Those data files of Stops and Demographics have been loaded in to the Postgres database on AWS.
<p>Run the next block for the access setup via Python

In [98]:
# SQL access functions as from your assignment
'''
takes a query and run it against a database on AWS
prints the result set
'''
def dbaccess(db,query):
    try:
        conn = pg.connect(host="52.15.99.183",database=db, 
                            user="rhodes", password="postpass")
        cur = conn.cursor()  # tie the cursor to the query
        cur.execute(query)  # within the cursor
        colnames = [desc[0] for desc in cur.description]
        print (colnames)
        nRows = 0
        for row in cur:    #walk thru the result set
            nRows += 1
            for v in row:
                print(v, end = '  ')
            print()
        print("{} tuples returned".format(nRows))
        
    except (Exception, pg.DatabaseError) as error :
        print ("Error while connecting to PostgreSQL", error)

    finally: #regardless of success or failure, clean up connection
    #closing database connection.
        if(conn):
            cur.close()
            conn.close()
            print("PostgreSQL connection is closed")

'''
takes a query and run it against a database on AWS
and returns the result set in a dataframe
'''
def db2df(db,query):
    try:
        conn = pg.connect(host="52.15.99.183",database=db, 
                            user="rhodes", password="postpass")
        datfr = pd.read_sql_query(query, conn)
        
    except (Exception, pg.DatabaseError) as error :
        print ("Error while connecting to PostgreSQL", error)

    finally: #regardless of success or failure, clean up connection
    #closing database connection.
        if(conn):
            conn.close()
            print("PostgreSQL connection is closed")
    return datfr

Answer the queries below against the data in the database.  The query here gives you a working retrieval and a template for your queries.

In [99]:
query = """ 
SELECT * 
FROM demog
"""
df = db2df("mpls",query)
df

PostgreSQL connection is closed


/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/ipykernel_1974/2605252322.py:40: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  datfr = pd.read_sql_query(query, conn)


,neighborhood,population,black,white,foreignborn,hhincome,poverty,collegegrad
0,Cedar Riverside,8247.0,0.353,0.464,0.408,18892.0,0.060,0.258
1,Phillips West,5184.0,0.199,0.538,0.318,18404.0,0.042,0.211
2,Downtown West,7141.0,0.561,0.211,0.203,67086.0,0.057,0.551
3,Downtown East,1674.0,0.543,0.221,0.221,70669.0,0.071,0.577
4,Shingle Creek,3249.0,0.407,0.259,0.140,59414.0,0.110,0.247
...,...,...,...,...,...,...,...,...
79,McKinley,3198.0,0.220,0.416,0.161,42556.0,0.101,0.122
80,Whittier,14604.0,0.549,0.187,0.215,35855.0,0.038,0.399
81,Lyndale,7441.0,0.515,0.226,0.285,38441.0,0.083,0.390
82,Columbia Park,1699.0,0.751,0.067,0.130,66545.0,0.058,0.418


Query #1: Retrieve just the attributes of neighborhoods, population, and income of only those neighborhoods predominately Black.  You just compare the proportions between Black and White to express the condition.

In [100]:
query = """ 
SELECT neighborhood, population, hhincome  
FROM demog
WHERE black>0.5
""" 
df = db2df("mpls",query)
df

PostgreSQL connection is closed


/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/ipykernel_1974/2605252322.py:40: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  datfr = pd.read_sql_query(query, conn)


,neighborhood,population,hhincome
0,Downtown West,7141.0,67086.0
1,Downtown East,1674.0,70669.0
2,Victory,4525.0,57148.0
3,Bottineau,1573.0,50900.0
4,Howe,6816.0,63492.0
...,...,...,...
59,Como,16022.0,67600.0
60,Whittier,14604.0,35855.0
61,Lyndale,7441.0,38441.0
62,Columbia Park,1699.0,66545.0


Query #2: Retrieve the attributes of date, problem, race, gender from just precinct 1, also limited to the months June, July and August. Have the results sorted by race.

In [101]:
query = """ 
SELECT date, problem, race, gender 
FROM stops
WHERE policeprecinct=1.0 AND date BETWEEN '2017-06-01' AND '2017-08-31'
""" 
df = db2df("mpls",query)
df

PostgreSQL connection is closed


/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/ipykernel_1974/2605252322.py:40: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  datfr = pd.read_sql_query(query, conn)


,date,problem,race,gender
0,2017-06-01 00:11:48,suspicious,East African,Male
1,2017-06-01 01:29:14,traffic,White,Male
2,2017-06-01 02:49:35,traffic,Black,Female
3,2017-06-01 03:23:13,suspicious,Black,Male
4,2017-06-01 03:39:48,suspicious,Unknown,Unknown
...,...,...,...,...
2025,2017-08-30 23:20:55,suspicious,White,Male
2026,2017-08-30 23:28:58,suspicious,Other,Female
2027,2017-08-30 23:29:39,traffic,White,Female
2028,2017-08-30 23:31:09,suspicious,NA,NA


Query #3:  The results of query #2 probably causes you to wonder how many of each race there are.  
<p>Copy and amend query #2 to create just a table of races and the count of each

In [102]:
query = """ 
SELECT race, COUNT(*)
FROM stops
WHERE policeprecinct=1.0 AND date BETWEEN '2017-06-01' AND '2017-08-31'
GROUP BY race
""" 
df = db2df("mpls",query)
df

PostgreSQL connection is closed


/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/ipykernel_1974/2605252322.py:40: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  datfr = pd.read_sql_query(query, conn)


,race,count
0,Latino,20
1,Black,564
2,East African,79
3,White,307
4,NA,711
5,Asian,12
6,Unknown,234
7,Other,34
8,Native American,69


Query #4-5: One of the likely tasks for data science is to create a csv file of a dump of the data in the database.  We have two tables and want to join the two tables so that we can make a single "flat file". In this case the demographic data needs to be joined to all of the stop records.
<p>Note the number of records in the stops table below.  When we join the demographic data, we want to be sure we don't lose any records in the join.

In [104]:
query = """ 
SELECT * 
FROM stops
""" 
df = db2df("mpls",query)
df

/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/ipykernel_1974/2605252322.py:40: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  datfr = pd.read_sql_query(query, conn)


PostgreSQL connection is closed


,caseid,idnum,date,problem,mdc,citationissued,personsearch,vehicesearch,prerace,race,gender,lat,long,policeprecinct,neighborhood
0,14049,17-060403,2017-02-19 05:46:18,traffic,other,NA,NA,NA,NA,NA,NA,44.948370,-93.242310,3.0,Corcoran
1,14050,17-060425,2017-02-19 07:38:24,suspicious,MDC,NA,NO,NO,Unknown,White,Male,44.952848,-93.289343,5.0,Lowry Hill East
2,14051,17-060434,2017-02-19 07:56:46,suspicious,MDC,NA,NO,NO,Unknown,White,Female,44.978459,-93.277331,1.0,Downtown West
3,14052,17-060439,2017-02-19 08:10:35,suspicious,MDC,NA,NO,NO,Unknown,Unknown,Unknown,44.986351,-93.237514,2.0,Marcy Holmes
4,14053,17-060449,2017-02-19 08:23:26,suspicious,other,NA,NA,NA,NA,NA,NA,45.008625,-93.290524,4.0,Hawthorne
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51915,14044,17-060385,2017-02-19 04:41:49,suspicious,MDC,NA,NO,NO,Unknown,White,Male,44.989100,-93.266040,2.0,Nicollet Island - East Bank
51916,14045,17-060386,2017-02-19 04:44:14,suspicious,MDC,NA,YES,YES,Unknown,Other,Male,44.954632,-93.257534,3.0,Midtown Phillips
51917,14046,17-060388,2017-02-19 04:46:39,suspicious,MDC,NA,YES,NO,Unknown,Black,Male,44.999666,-93.306949,4.0,Willard - Hay
51918,14047,17-060392,2017-02-19 04:58:27,traffic,MDC,NA,NO,NO,Unknown,Black,Female,45.012667,-93.255515,2.0,Holland


Query #4:  Just do an inner (or natural) join of the two tables.  How many rows do you get?

In [105]:
query = """ 
SELECT stops.*, demog.* 
FROM stops
JOIN demog 
ON demog.neighborhood=stops.neighborhood
""" 
df = db2df("mpls",query)
df

/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/ipykernel_1974/2605252322.py:40: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  datfr = pd.read_sql_query(query, conn)


PostgreSQL connection is closed


,caseid,idnum,date,problem,mdc,citationissued,personsearch,vehicesearch,prerace,race,...,policeprecinct,neighborhood,neighborhood,population,black,white,foreignborn,hhincome,poverty,collegegrad
0,14049,17-060403,2017-02-19 05:46:18,traffic,other,NA,NA,NA,NA,NA,...,3.0,Corcoran,Corcoran,4339.0,0.497,0.136,0.151,54722.0,0.071,0.329
1,14050,17-060425,2017-02-19 07:38:24,suspicious,MDC,NA,NO,NO,Unknown,White,...,5.0,Lowry Hill East,Lowry Hill East,6357.0,0.760,0.105,0.145,49868.0,0.066,0.569
2,14051,17-060434,2017-02-19 07:56:46,suspicious,MDC,NA,NO,NO,Unknown,White,...,1.0,Downtown West,Downtown West,7141.0,0.561,0.211,0.203,67086.0,0.057,0.551
3,14052,17-060439,2017-02-19 08:10:35,suspicious,MDC,NA,NO,NO,Unknown,Unknown,...,2.0,Marcy Holmes,Marcy Holmes,10496.0,0.735,0.058,0.174,27104.0,0.042,0.587
4,14053,17-060449,2017-02-19 08:23:26,suspicious,other,NA,NA,NA,NA,NA,...,4.0,Hawthorne,Hawthorne,4609.0,0.186,0.456,0.167,21936.0,0.062,0.151
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49615,14044,17-060385,2017-02-19 04:41:49,suspicious,MDC,NA,NO,NO,Unknown,White,...,2.0,Nicollet Island - East Bank,Nicollet Island - East Bank,1393.0,0.782,0.031,0.138,83520.0,0.076,0.734
49616,14045,17-060386,2017-02-19 04:44:14,suspicious,MDC,NA,YES,YES,Unknown,Other,...,3.0,Midtown Phillips,Midtown Phillips,4949.0,0.246,0.297,0.360,46055.0,0.135,0.236
49617,14046,17-060388,2017-02-19 04:46:39,suspicious,MDC,NA,YES,NO,Unknown,Black,...,4.0,Willard - Hay,Willard - Hay,9074.0,0.215,0.516,0.141,44733.0,0.048,0.269
49618,14047,17-060392,2017-02-19 04:58:27,traffic,MDC,NA,NO,NO,Unknown,Black,...,2.0,Holland,Holland,5016.0,0.472,0.193,0.262,45343.0,0.057,0.233


I get 49,620 rows.

You should have come up short.  The fix is to use a left outer join.
<p>Query #5: Do a left outer join and keep all of the attributes.  <b>Unfortunately you did not have a homework problem to exercise this, but it's in the notes.</b>  While you don't have to do it here, the dataframe could be output to a csv file and you're done!

In [106]:
query = """ 
SELECT stops.*, demog.* 
FROM stops
LEFT OUTER JOIN demog 
ON demog.neighborhood=stops.neighborhood
""" 
df = db2df("mpls",query)
df

/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/ipykernel_1974/2605252322.py:40: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  datfr = pd.read_sql_query(query, conn)


PostgreSQL connection is closed


,caseid,idnum,date,problem,mdc,citationissued,personsearch,vehicesearch,prerace,race,...,policeprecinct,neighborhood,neighborhood,population,black,white,foreignborn,hhincome,poverty,collegegrad
0,14049,17-060403,2017-02-19 05:46:18,traffic,other,NA,NA,NA,NA,NA,...,3.0,Corcoran,Corcoran,4339.0,0.497,0.136,0.151,54722.0,0.071,0.329
1,14050,17-060425,2017-02-19 07:38:24,suspicious,MDC,NA,NO,NO,Unknown,White,...,5.0,Lowry Hill East,Lowry Hill East,6357.0,0.760,0.105,0.145,49868.0,0.066,0.569
2,14051,17-060434,2017-02-19 07:56:46,suspicious,MDC,NA,NO,NO,Unknown,White,...,1.0,Downtown West,Downtown West,7141.0,0.561,0.211,0.203,67086.0,0.057,0.551
3,14052,17-060439,2017-02-19 08:10:35,suspicious,MDC,NA,NO,NO,Unknown,Unknown,...,2.0,Marcy Holmes,Marcy Holmes,10496.0,0.735,0.058,0.174,27104.0,0.042,0.587
4,14053,17-060449,2017-02-19 08:23:26,suspicious,other,NA,NA,NA,NA,NA,...,4.0,Hawthorne,Hawthorne,4609.0,0.186,0.456,0.167,21936.0,0.062,0.151
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51915,14044,17-060385,2017-02-19 04:41:49,suspicious,MDC,NA,NO,NO,Unknown,White,...,2.0,Nicollet Island - East Bank,Nicollet Island - East Bank,1393.0,0.782,0.031,0.138,83520.0,0.076,0.734
51916,14045,17-060386,2017-02-19 04:44:14,suspicious,MDC,NA,YES,YES,Unknown,Other,...,3.0,Midtown Phillips,Midtown Phillips,4949.0,0.246,0.297,0.360,46055.0,0.135,0.236
51917,14046,17-060388,2017-02-19 04:46:39,suspicious,MDC,NA,YES,NO,Unknown,Black,...,4.0,Willard - Hay,Willard - Hay,9074.0,0.215,0.516,0.141,44733.0,0.048,0.269
51918,14047,17-060392,2017-02-19 04:58:27,traffic,MDC,NA,NO,NO,Unknown,Black,...,2.0,Holland,Holland,5016.0,0.472,0.193,0.262,45343.0,0.057,0.233


<h3>Regular Expressions</h3>
<p>Set up to read the Minneapolis Stops file as one long text string and then write the requested regular expressions to search the file.
<p> You may need to change the filename depending on where you saved the data file as you used above.

In [107]:
import re
inf= open('/Users/Charley/Dropbox/Charley/Juniata/Data Science Masters/Data Acquisition & Visualization/MplsStops(1).csv','r',encoding='utf-8')
#just input the whole file in 'tweets' as a string
stops = inf.read()
inf.close()

Regex #1: Find the number lines in the data that contain the string "Unknown" or "Other". Use only one regular expression.

In [108]:
regex = re.compile(r'(Unknown)|(Other)') #insert regex inside the single quotes
len(regex.findall(stops))

45880

Regex #2: Find the number lines in the data that contain the pattern "Black" or "African" followed by "Male". Use only one regular expression.

In [109]:
regex = re.compile(r'((Black)|(African)), Male') #insert regex inside the single quotes
len(regex.findall(stops))

0

Regex #3: Write a regular expression that matches the idNum (2 digits, a dash, then 6 digits). It should match every line. Use only one regular expression.

In [110]:
regex = re.compile(r'\d\d-\d\d\d\d\d\d') #insert regex inside the single quotes
len(regex.findall(stops))

51920